# Export the data in a custom format
To resolve the searched lists of bibcodes for the publications and references into actual metadata, we have to call the API for each bibcode and extract the specific information we want.

## Imports

In [3]:
import json
import requests
import time
import pandas as pd
import pickle
from dotenv import load_dotenv
import os
from tqdm import tqdm
import numpy as np
import re
import html
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import math

# Load the environment variables from the .env file
load_dotenv()

# Access to the environment variable
token = os.getenv('ADS_API_KEY')

# --- Configuration Constants ---
ADS_EXPORT_URL = "https://api.adsabs.harvard.edu/v1/export/custom"
MAX_BIBCODES_PER_REQUEST = 2000  # ADS API limit
DEFAULT_MAX_RETRIES = 5
DEFAULT_BACKOFF_FACTOR = 2
REQUEST_TIMEOUT_SECONDS = 120
MAX_CONCURRENT_WORKERS = 5  # Conservative to avoid rate limiting

# Data paths (aligned with ADS_search.ipynb)
DATA_DIR = Path("2_data")
INPUT_FILE = DATA_DIR / "bibcodes_references_20260120_124208.pkl"

## Load data
Let's first load the data from the pickle file we saved earlier.

In [4]:
# Load the variables from the latest pickle file (aligned with ADS_search.ipynb output)
if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_FILE}. Run ADS_search.ipynb first.")

with open(INPUT_FILE, "rb") as f:
    bibcodes, references, esources, fulltext_urls = pickle.load(f)

print(f"Loaded {len(bibcodes)} bibcodes from {INPUT_FILE}")

Loaded 180381 bibcodes from 2_data\bibcodes_references_20260120_124208.pkl


## Exporting the returned list of Bibcodes using a custom format
The options for exporting data, are provides in the [full API documentation](https://ui.adsabs.harvard.edu/help/api/api-docs.html). For our case, we dont need the standard but a custom format, as we specifically want to control the returned information in each export. We also want a compareable format to the Web of Science Format to have high compatibility with many tools. In the cell bellow you see the mapping from the ADS Field Specifiers to the WOS Field Tags. More information about custom export formats can be found [here](https://ui.adsabs.harvard.edu/help/actions/export#the-ads-custom-format).

We use a custom format, which is arranged into a readable bibliography-style format (see [Field Specifiers](https://ui.adsabs.harvard.edu/help/actions/export#:~:text=equivalent%20to%20%251H.-,other%20formats,-Specifier)):

`%RxOx%ExOx%TxOx%YxOx%JxOx%SxOx%VxOx%p xOx%P xOx%BxOx%KxOx%dxOx%FxOx%WxOx%cxOx`

The base URL `https://api.adsabs.harvard.edu/v1/export/custom` utilizes the POST HTTP method. The payload needs both the list of bibcodes to export and the custom format string. We define the our format as:

In [5]:
# ADS Field Specifier → Description (WOS Field Tag: Description)

# %R → Bibliographic Code (UT: Accession Number)
# %E → Author (AU: Authors)
# %T → Title (TI: Document Title)
# %Y → Year (PY: Year Published)
# %J → Journal (SO: Publication Name)
# %q → Journal abbreviation (JI: ISO Source Abbreviation)
# %S → Issue (IS: Issue)
# %V → Volume (VL: Volume)
# %p → First Page (BP: Beginning Page)
# %P → Last Page (EP: Ending Page)
# %B → Abstract (AB: Abstract)
# %K → Keywords (DE: Author Keywords)
# %d → DOI (DI: Digital Object Identifier)
# %F → Author Affiliation (C1: Author Address / RP: Reprint Address)
# %W → Category (DT: Document Type)
# %c → Citation Count (TC: Times Cited Count)

custom_format = "%RxOx%ExOx%TxOx%YxOx%JxOx%qxOx%SxOx%VxOx%p xOx%P xOx%BxOx%KxOx%dxOx%FxOx%WxOx%cxOx"

Let's now define the function which will export the results of our query in the custom format we defined above. We'll use the requests package to send the POST request to the API. The payload is a dictionary containing the list of bibcodes and the custom format string. We'll also specify the sort order as ADS Score, descending (by default the returned references are sorted by score first).

In [6]:
def create_ads_session(token):
    """Create a persistent HTTP session with ADS authentication."""
    session = requests.Session()
    session.headers.update({
        'Authorization': f'Bearer {token}',
        'Content-Type': 'application/json'
    })
    return session


def export_single_chunk(session, bibcodes_chunk, chunk_index, custom_format,
                        max_retries=DEFAULT_MAX_RETRIES,
                        backoff_factor=DEFAULT_BACKOFF_FACTOR):
    """
    Export a single chunk of bibcodes. Thread-safe.
    
    Returns:
        tuple: (chunk_index, export_data, error_message)
    """
    payload = {'bibcode': bibcodes_chunk, 'format': custom_format, 'sort': 'score desc'}

    for attempt in range(max_retries + 1):
        try:
            response = session.post(ADS_EXPORT_URL, data=json.dumps(payload),
                                    timeout=REQUEST_TIMEOUT_SECONDS)

            if response.status_code == 200:
                return (chunk_index, response.json()['export'], None)

            # Handle rate limiting with Retry-After header
            if response.status_code == 429:
                wait = int(response.headers.get("Retry-After", 60))
                print(f"\n[Chunk {chunk_index}] Rate limited. Waiting {wait}s...")
                time.sleep(wait)
                continue

            # Handle server errors with exponential backoff
            if response.status_code >= 500:
                if attempt < max_retries:
                    sleep_time = backoff_factor * (2 ** attempt)
                    print(f"\n[Chunk {chunk_index}] Server error {response.status_code}. "
                          f"Retry {attempt+1}/{max_retries} in {sleep_time}s...")
                    time.sleep(sleep_time)
                    continue

            # Handle 400 errors (usually bad bibcodes)
            if response.status_code == 400:
                try:
                    err_detail = response.json().get('error', response.text[:200])
                except:
                    err_detail = response.text[:200]
                print(f"\n--- API 400 BAD REQUEST [Chunk {chunk_index}] ---")
                print(f"Error Detail: {err_detail}")
                return (chunk_index, None, f"400: {err_detail}")

            response.raise_for_status()

        except requests.exceptions.Timeout:
            if attempt >= max_retries:
                return (chunk_index, None, "Timeout after all retries")
            sleep_time = backoff_factor * (2 ** attempt)
            print(f"\n[Chunk {chunk_index}] Timeout. Retry {attempt+1}/{max_retries} in {sleep_time}s...")
            time.sleep(sleep_time)
            
        except requests.exceptions.RequestException as e:
            if attempt >= max_retries:
                return (chunk_index, None, str(e))
            sleep_time = backoff_factor * (2 ** attempt)
            time.sleep(sleep_time)

    return (chunk_index, None, "Max retries exceeded")


def export_bibcodes(bibcodes_list, max_records=MAX_BIBCODES_PER_REQUEST,
                    max_retries=DEFAULT_MAX_RETRIES, backoff_factor=DEFAULT_BACKOFF_FACTOR,
                    max_workers=MAX_CONCURRENT_WORKERS):
    """
    Export bibcodes in parallel chunks using the ADS custom export API.
    
    Args:
        bibcodes_list: List of bibcodes to export
        max_records: Maximum bibcodes per API request (default: 2000, ADS limit)
        max_retries: Maximum retry attempts per chunk
        backoff_factor: Exponential backoff multiplier
        max_workers: Number of concurrent API requests
    
    Returns:
        str: Concatenated export data for all bibcodes
    """
    if not bibcodes_list:
        return ""

    # Create chunks using list comprehension (cleaner than manual loop)
    chunks = [bibcodes_list[i:i + max_records]
              for i in range(0, len(bibcodes_list), max_records)]
    num_chunks = len(chunks)

    # Create persistent session (reuses TCP connections)
    session = create_ads_session(token)
    
    # Pre-allocate results list to maintain order
    results = [None] * num_chunks
    errors = []

    print(f"Exporting {len(bibcodes_list)} bibcodes in {num_chunks} chunks with {max_workers} workers...")
    pbar = tqdm(total=len(bibcodes_list), desc="Exporting bibcodes")

    # Process chunks in parallel
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all chunks
        future_to_chunk = {
            executor.submit(export_single_chunk, session, chunk, idx, custom_format,
                          max_retries, backoff_factor): (idx, len(chunk))
            for idx, chunk in enumerate(chunks)
        }

        # Collect results as they complete
        for future in as_completed(future_to_chunk):
            chunk_idx, chunk_size = future_to_chunk[future]
            try:
                idx, data, error = future.result()
                if error:
                    errors.append((idx, error))
                else:
                    results[idx] = data
                pbar.update(chunk_size)
            except Exception as e:
                errors.append((chunk_idx, str(e)))
                pbar.update(chunk_size)

    pbar.close()
    session.close()

    # Report errors
    if errors:
        print(f"\n{len(errors)} chunk(s) failed:")
        for idx, err in errors[:5]:  # Show first 5 errors
            print(f"  Chunk {idx}: {err}")
        if len(errors) > 5:
            print(f"  ... and {len(errors) - 5} more errors")

    # Concatenate results (filter out None from failed chunks)
    # Using list + join is O(n) instead of O(n²) string concatenation
    valid_results = [r for r in results if r is not None]
    
    if len(valid_results) < num_chunks:
        print(f"\nWarning: {num_chunks - len(valid_results)} chunks failed.")

    return ''.join(valid_results)

We can now use the function to export the results of our query in the custom format we defined above. For exports above 100000 publications this can take a while (above 10 min).

In [7]:
# For the first use case, pass the bibcodes list to the function and store the returned data.
data = export_bibcodes(bibcodes)

Exporting 180381 bibcodes in 91 chunks with 5 workers...


Exporting bibcodes:   0%|          | 0/180381 [00:00<?, ?it/s]

Exporting bibcodes:  53%|█████▎    | 96000/180381 [02:49<01:47, 783.50it/s] 


[Chunk 50] Server error 502. Retry 1/5 in 2s...


Exporting bibcodes: 100%|██████████| 180381/180381 [04:46<00:00, 629.54it/s] 


We check again, if the number of bibcodes ist still the same.

In [8]:
print('Number of exported Bibcodes:', int(data.count('xOx')/16))

Number of exported Bibcodes: 180359


Let's look at the content of the returned data.

In [9]:
data[0:1000]

'2000MNRAS.319..168CxOxCole S, Lacey CG, Baugh CM, Frenk CSxOxHierarchical galaxy formationxOx2000xOxMonthly Notices of the Royal Astronomical SocietyxOxMNRASxOx1xOx319xOx168 xOx204 xOxWe describe the GALFORM semi-analytic model for calculating the formation and evolution of galaxies in hierarchical clustering cosmologies. It improves upon, and extends, the earlier scheme developed by Cole et al. The model employs a new Monte Carlo algorithm to follow the merging evolution of dark matter haloes with arbitrary mass resolution. It incorporates realistic descriptions of the density profiles of dark matter haloes and the gas they contain; it follows the chemical evolution of gas and stars, and the associated production of dust; and it includes a detailed calculation of the sizes of discs and spheroids. Wherever possible, our prescriptions for modelling individual physical processes are based on results of numerical simulations. They require a number of adjustable parameters, which we fix b

It comes back as a single string, so our next step will be to divide it into individual strings, each representing a specific paper. Furthermore, we can segment each of these strings into a collection of fields, which correspond to each field in the custom format string. With these field names, we can create a dictionary for each paper or a dataframe for the whole set. Before that we repeat the same process as above for the references.

## Exporting the returned list of Bibcodes of the References using a custom format
To be able to perform the same kind of export as above, we first have to flatten the references, which is a list of lists at the moment. After that we remove duplicates, to reduce the number of requested objects.

In [10]:
# We flatten the list to "remove the brackets within the list" using a nested list comprehension. This will un-nest each list stored in our list of lists.
flattened_references = [val for sublist in references for val in sublist]

# We create a dictionary, using the List items as keys. This will automatically remove any duplicates because dictionaries cannot have duplicate keys.
# Then, we convert the dictionary back into a list.
single_references = list(dict.fromkeys(flattened_references))

print("Number of unique references: " + str(len(single_references)))
print("Total Number of references: " + str(len(flattened_references)))

Number of unique references: 442919
Total Number of references: 1986991


We perform the same steps as above, but this time we use the references list as input for the export function. As the number of references is larger than the number of publications, this will take even longer.

In [11]:
# Pass the single_references list to the function and store the returned data.
data_references = export_bibcodes(single_references)

Exporting 442919 bibcodes in 222 chunks with 5 workers...


Exporting bibcodes: 100%|██████████| 442919/442919 [10:02<00:00, 735.36it/s] 


In [12]:
print('Number of exported Bibcodes of the References:', int(data_references.count('xOx')/16))

Number of exported Bibcodes of the References: 442919


In [13]:
data_references[0:1000]

'1998AJ....116.1009RxOxRiess AG, Filippenko AV, Challis P, Clocchiatti A, Diercks A, Garnavich PM, Gilliland RL, Hogan CJ, Jha S, Kirshner RP, Leibundgut B, Phillips MM, Reiss D, Schmidt BP, Schommer RA, Smith RC, Spyromilio J, Stubbs C, Suntzeff NB, Tonry JxOxObservational Evidence from Supernovae for an Accelerating Universe and a Cosmological ConstantxOx1998xOxThe Astronomical JournalxOxAJxOx3xOx116xOx1009 xOx1038 xOxWe present spectral and photometric observations of 10 Type Ia supernovae (SNe Ia) in the redshift range 0.16 <= z <= 0.62. The luminosity distances of these objects are determined by methods that employ relations between SN Ia luminosity and light curve shape. Combined with previous data from our High-z Supernova Search Team and recent results by Riess et al., this expanded set of 16 high-redshift supernovae and a set of 34 nearby supernovae are used to place constraints on the following cosmological parameters: the Hubble constant (H_0), the mass density (Omega_M), th

We now can create a dataframe of all papers of our whole set and their references.

## Transforming the exported Publications and their References
Next we will transform the exported data into a pandas Dataframe structure, to enable further analysis and to be able to easyly define a useful output.

### Queried Publications

In [14]:
def process_data(data):
    """
    Function to process raw data into a structured pandas DataFrame

    Parameters:
    data (str): The raw data to be processed

    Returns:
    queried_publications (DataFrame): The processed data as a pandas DataFrame
    """

    # Split the returned data into individual records, then split each record into its fields
    # Note: we're splitting on 'xOx\n' instead of just '\n' to avoid splitting within fields that contain line breaks
    exported_bibcodes = [x.split('xOx') for x in data.split('xOx\n')]

    # Convert the list of records into a pandas DataFrame and remove the last row and give the columns names
    queried_publications = pd.DataFrame(exported_bibcodes)[:-1]
    queried_publications.columns = ['Bibcode', 'Author', 'Title', 'Year', 'Journal', 'Journal Abbreviation', 'Issue', 'Volume', 'First Page', 'Last Page', 'Abstract', 'Keywords', 'DOI', 'Affiliation', 'Category', 'Citation Count']

    # Replace all newline characters within the DataFrame with spaces, removing unwanted line breaks
    queried_publications = queried_publications.replace(r'\n',' ', regex=True)

    # Set the DataFrame's index to be the 'Bibcode' column
    # queried_publications.set_index("Bibcode", inplace=True)

    return queried_publications

We apply `process_data` to the list of papers we exported above.

In [15]:
queried_publications = process_data(data)
queried_publications.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180361 entries, 0 to 180360
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   Bibcode               180361 non-null  object
 1   Author                180361 non-null  object
 2   Title                 180361 non-null  object
 3   Year                  180361 non-null  object
 4   Journal               180361 non-null  object
 5   Journal Abbreviation  180361 non-null  object
 6   Issue                 180359 non-null  object
 7   Volume                180359 non-null  object
 8   First Page            180359 non-null  object
 9   Last Page             180359 non-null  object
 10  Abstract              180357 non-null  object
 11  Keywords              180357 non-null  object
 12  DOI                   180357 non-null  object
 13  Affiliation           180357 non-null  object
 14  Category              180357 non-null  object
 15  Citation Count   

Before we write the resolved queried publications into a resuable format, we add the list of reference Bibcodes and esource urls of each publications to the dataframe.

In [16]:
# Create a combined dictionary with Bibcode, References, and PDF_URL
combineddict = {
    'Bibcode': bibcodes,
    'References': references,
    'PDF_URL': fulltext_urls  # Aligned with ADS_search.ipynb variable name
}

# Convert the combined dictionary into a pandas DataFrame
combineddict_df = pd.DataFrame(combineddict)

# Perform a merge operation to add 'References' and 'PDF_URL' columns to 'queried_publications'
queried_publications = pd.merge(queried_publications, combineddict_df[['Bibcode', 'References', 'PDF_URL']], on='Bibcode', how='left')

In [17]:
queried_publications.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180361 entries, 0 to 180360
Data columns (total 18 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   Bibcode               180361 non-null  object
 1   Author                180361 non-null  object
 2   Title                 180361 non-null  object
 3   Year                  180361 non-null  object
 4   Journal               180361 non-null  object
 5   Journal Abbreviation  180361 non-null  object
 6   Issue                 180359 non-null  object
 7   Volume                180359 non-null  object
 8   First Page            180359 non-null  object
 9   Last Page             180359 non-null  object
 10  Abstract              180357 non-null  object
 11  Keywords              180357 non-null  object
 12  DOI                   180357 non-null  object
 13  Affiliation           180357 non-null  object
 14  Category              180357 non-null  object
 15  Citation Count   

The last step is cleaning the dataframe, by removing duplicates, html markup and NaN values.

In [18]:
import pandas as pd
import numpy as np
import html
import re

def clean_dataframe(df):
    """
    This function takes a DataFrame and performs several cleaning operations:
    1. Replaces '[]' in the 'References' column with NaN.
    2. Drops all rows with NaN in the 'Year' column and converts the column to integer format.
    3. Fills all NaN values in the 'References' column with an empty string.
    4. Cleans HTML tags and entities from the 'Title', 'Abstract', and 'full_text' columns.
    5. Removes all '-' in the 'Author' column.
    6. Keeps only the first number in ranges and removes all whitespaces in the 'Issue', 'Volume', 'First Page', 'Last Page' columns.

    Args:
    df (pandas.DataFrame): Input DataFrame.

    Returns:
    df (pandas.DataFrame): Processed DataFrame.
    """

    # Create a deep copy of the DataFrame to avoid modifying the original DataFrame
    df = df.copy()

    # Function to unescape HTML entities and remove HTML tags
    def clean_html(text):
        if isinstance(text, str):  # Only apply if the input is a string
            text = html.unescape(text)
            text = re.sub('<.*?>', '', text)
        return text

    # Function to clean range values and remove whitespaces
    def clean_range(value):
        if isinstance(value, str):  # Only apply if the input is a string
            value = value.split('-')[0]  # Keep only the first number in ranges
            value = value.replace(" ", "")  # Remove all whitespaces
        return value

    # Replace '[]' in the 'References' column with NaN and fill NaN values with an empty string
    if 'References' in df.columns:
        df['References'] = df['References'].apply(lambda x: np.nan if x == '[]' else x)
        df['References'] = df['References'].fillna("")

    # Remove rows with non-numeric 'Year' values and convert 'Year' to int
    df = df[pd.to_numeric(df['Year'], errors='coerce').notnull()]
    df['Year'] = df['Year'].astype(int)

    # Clean HTML tags and entities from the 'Title', 'Abstract', and 'full_text' columns
    df['Title'] = df['Title'].apply(clean_html)
    df['Abstract'] = df['Abstract'].apply(clean_html)
    if 'full_text' in df.columns:
        df['full_text'] = df['full_text'].apply(clean_html)

    # Remove all '-' in the 'Author' column
    if 'Author' in df.columns:
        df['Author'] = df['Author'].str.replace('-', '')

    # Clean range values and remove whitespaces in the 'Issue', 'Volume', 'First Page', 'Last Page' columns
    # NOTE: DOI removed from this list - DOIs can contain hyphens (e.g., "10.1007/978-3-319-12345-6")
    columns_to_clean = ['Issue', 'Volume', 'First Page', 'Last Page']
    for column in columns_to_clean:
        if column in df.columns:
            df[column] = df[column].apply(clean_range)

    return df

In [19]:
#| column: page
#| label: Clean Data

queried_publications = clean_dataframe(queried_publications)
queried_publications

,Bibcode,Author,Title,Year,Journal,Journal Abbreviation,Issue,Volume,First Page,Last Page,Abstract,Keywords,DOI,Affiliation,Category,Citation Count,References,PDF_URL
0,2000MNRAS.319..168C,"Cole S, Lacey CG, Baugh CM, Frenk CS",Hierarchical galaxy formation,2000,Monthly Notices of the Royal Astronomical Society,MNRAS,1,319,168,204,We describe the GALFORM semi-analytic model fo...,"GALAXIES: FORMATION, Astrophysics",10.1046/j.1365-8711.2000.03879.x,"AA(Durham University, Department of Physics); ...",article,1811,"[1955ApJ...121..161S, 1972A&A....20..383T, 197...",https://ui.adsabs.harvard.edu/link_gateway/200...
1,2000A&A...363.1081C,Claret A,A new non-linear limb-darkening law for LTE st...,2000,Astronomy and Astrophysics,A&A,,363,1081,1190,"A new non-linear limb-darkening law, based on ...","STARS: ATMOSPHERES, STARS: BINARIES: ECLIPSING",,AA(Institute of Astrophysics of Andalusia),article,1115,"[1921MNRAS..81..361M, 1965BAICz..16..195G, 197...",https://ui.adsabs.harvard.edu/link_gateway/200...
2,2000PhRvL..85.5042P,"Parikh MK, Wilczek F",Hawking Radiation As Tunneling,2000,Physical Review Letters,PhRvL,24,85,5042,5045,We present a short and direct derivation of Ha...,"High Energy Physics - Theory, General Relativi...",10.1103/PhysRevLett.85.5042,"AA(Princeton University, Department of Physics...",article,1797,"[1921CR....173..677P, 1975CMaPh..43..199H, 197...",None
3,2000PhRvL..85.4438A,"ArmendarizPicon C, Mukhanov V, Steinhardt PJ",Dynamical Solution to the Problem of a Small C...,2000,Physical Review Letters,PhRvL,21,85,4438,4441,Increasing evidence suggests that most of the ...,"Astrophysics, General Relativity and Quantum C...",10.1103/PhysRevLett.85.4438,"AA(University of Munich, Department of Physics...",article,1897,"[1986NuPhB.277....1G, 1988ApJ...325L..17P, 198...",None
4,2000ApJ...542..464C,"Chabrier G, Baraffe I, Allard F, Hauschildt P",Evolutionary Models for Very Low-Mass Stars an...,2000,The Astrophysical Journal,ApJ,1,542,464,472,We present evolutionary calculations for very ...,"Hertzsprung-Russell, Stars: Evolution, Stars: ...",10.1086/309513,"AA(Observatoire de Lyon, Centre Recherche Astr...",article,1199,"[1969ApJS...18..297H, 1978Icar...36....1R, 198...",None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
180356,1911Natur..88....6S,Stromeyer CE,"Solar Eclipse—April, 1912",1911,Nature,Natur,2192,88,6,,A FEW years ago I read a short paper before th...,,10.1038/088006b0,"AA(""Lancefield"", West Didsbury.)",article,0,[],None
180357,1918Obs....41..321S,Silberstein L,Einstein's theory of gravitation,1918,The Observatory,Obs,,41,321,323,,,,AA(-),article,0,[],None
180358,1919giao.book.....H,Hosmer GL,"Geodesy, including astronomical observations, ...",1919,New York,giao.book,,,,,,GEODESY,,AA(-),book,0,[],None
180359,1913Natur..91..424S,Schiller FCS,Radio-Activity and the Age of the Earth,1913,Nature,Natur,2278,91,424,,"MR. HOLMES, in his interesting letter in NATUR...",,10.1038/091424b0,"AA(Corpus Christi College, Oxford, June 23.)",article,0,[],None




We finally have our data in useful format and write the resulting Dataframe into a JSON-File.

In [20]:
# Write the resulting Publications into a JSON-File.
queried_publications.to_json('2_data/queried_publications.json', orient='records', lines=True)

### Filter resulting Dataframe
If we only want part of the data, let's say only the papers by a certain Author, we can filter the dataframe accordingly.

In [21]:
# Filter the DataFrame to keep only rows with 'Treder' in the 'Author' column
queried_publications_filtered = queried_publications[queried_publications['Author'].str.contains('Treder')]

# Write the filtered DataFrame to a JSON file
queried_publications_filtered.to_json('2_data/queried_publications_filtered.json', orient='records', lines=True)

### References of queried Publications
We repeat the same steps as above, now for the references.

In [22]:
queried_references = process_data(data_references)
queried_references.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 442920 entries, 0 to 442919
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   Bibcode               442920 non-null  object
 1   Author                442920 non-null  object
 2   Title                 442920 non-null  object
 3   Year                  442920 non-null  object
 4   Journal               442920 non-null  object
 5   Journal Abbreviation  442920 non-null  object
 6   Issue                 442919 non-null  object
 7   Volume                442919 non-null  object
 8   First Page            442919 non-null  object
 9   Last Page             442919 non-null  object
 10  Abstract              442918 non-null  object
 11  Keywords              442918 non-null  object
 12  DOI                   442918 non-null  object
 13  Affiliation           442918 non-null  object
 14  Category              442918 non-null  object
 15  Citation Count   

We apply `clean_dataframe` also to query_references.

In [23]:
queried_references = clean_dataframe(queried_references)

And again we write the resulting Dataframe into a JSON-File. But since we are writing the references, we don't add the list of reference Bibcodes of each publications to the dataframe.

In [24]:
# Write the resulting Publications into a JSON-File.
queried_references.to_json('2_data/queried_references.json', orient='records', lines=True)